# Karman Nowcasting Model — Inference Demo

This notebook walks through running inference with a pre-trained **Karman nowcasting model** that predicts thermospheric neutral density (kg/m³) from space weather and orbital parameters.

## How it works

The model is a simple feedforward neural network (MLP) that learns a **residual correction** on top of an exponential atmosphere baseline:

$$\rho_{\text{pred}} = \text{unscale}\!\Big(\text{scale}(\rho_{\text{expo}}) + \tanh\!\big(\text{NN}(\mathbf{x})\big)\Big)$$

where:
- $\rho_{\text{expo}}$ is the density from a simple exponential atmosphere model (a rough first guess),
- $\mathbf{x}$ is a vector of 18 space-weather and orbital features (already normalized),
- The NN outputs a correction in normalized log-space, squeezed through $\tanh$ to keep it bounded,
- `scale` / `unscale` convert between physical density and the normalized $[-1, 1]$ log-space used during training.

## 1. Imports and setup

In [ ]:
import sys, os, json
import torch
import matplotlib.pyplot as plt

# Make the karman package importable from the notebooks/ directory
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from karman.nn import SimpleNetwork

torch.set_default_dtype(torch.float32)

## 2. Load sample data

The demo data (`demo_samples.json`) contains:
- **`normalization_dict`** — min/max of log₁₀(density) used to map densities to [−1, 1] during training.
- **`feature_names`** — the 18 input features (altitude, latitude, solar indices, geomagnetic indices, and cyclical encodings of longitude / day-of-year / time-of-day).
- **`samples`** — a list of pre-processed data points, each with:
  - `instantaneous_features`: the 18 normalized input values,
  - `exponential_atmosphere`: the baseline exponential-model density (kg/m³),
  - `ground_truth`: the observed density (kg/m³).

In [ ]:
SAMPLES_PATH = os.path.join('..', 'scripts', 'demo_samples.json')

with open(SAMPLES_PATH) as f:
    data = json.load(f)

log_min = data['normalization_dict']['log_exp_residual']['min']
log_max = data['normalization_dict']['log_exp_residual']['max']
feature_names = data['feature_names']
samples = data['samples']

print(f"Normalization range: log10(density) ∈ [{log_min:.4f}, {log_max:.4f}]")
print(f"Number of features: {len(feature_names)}")
print(f"Number of samples:  {len(samples)}")
print(f"\nFeatures:")
for i, name in enumerate(feature_names):
    print(f"  [{i:2d}] {name}")

## 3. Define the scaling functions

Because density spans many orders of magnitude (~10⁻¹⁴ to 10⁻¹⁰ kg/m³), the model works in **normalized log-space**. Training targets are mapped to [−1, 1] via:

$$\text{scaled} = 2 \cdot \frac{\log_{10}(\rho) - \log_{\min}}{\log_{\max} - \log_{\min}} - 1$$

and the inverse:

$$\rho = 10^{\;\frac{(\log_{\max} - \log_{\min})(\text{scaled} + 1)}{2} \;+\; \log_{\min}}$$

In [ ]:
def scale_density(density, log_min, log_max):
    """Map a physical density (kg/m³) into normalized log-space [-1, 1]."""
    tmp = torch.log10(density)
    return 2.0 * (tmp - log_min) / (log_max - log_min) - 1.0


def unscale_density(scaled, log_min, log_max):
    """Map a normalized log-space value back to physical density (kg/m³)."""
    tmp = (log_max - log_min) * (scaled + 1) / 2 + log_min
    return torch.pow(10, tmp)


# Quick sanity check: round-trip a known density value
test_rho = torch.tensor(1.5e-12)
reconstructed = unscale_density(scale_density(test_rho, log_min, log_max), log_min, log_max)
print(f"Original:      {test_rho.item():.6e}")
print(f"Round-tripped:  {reconstructed.item():.6e}")
print(f"Match: {torch.allclose(test_rho, reconstructed)}")

## 4. Load the pre-trained model from S3

The model weights are hosted on a **public S3 bucket** — no AWS credentials required. We download them to a local cache on first run using `urllib`.

The model is a `SimpleNetwork` — a fully-connected MLP with 3 hidden layers of 128 units each and LeakyReLU activations:

```
Input (18) → [Linear → LeakyReLU] × 3 → Linear → Output (1)
```

In [ ]:
import io
from urllib.request import urlopen

MODEL_URL = (
    "https://nasa-radiant-data.s3.amazonaws.com/"
    "helioai-datasets/hl-therm/models/"
    "karman_nowcast_model_log_exp_residual_valid_mape_15.14_params_35585.torch"
)
HIDDEN_LAYER_DIM = 128
HIDDEN_LAYERS = 3

# Download weights into memory (small model — ~140 KB)
print("Downloading model weights from S3 …")
with urlopen(MODEL_URL) as resp:
    model_bytes = resp.read()
print(f"Downloaded {len(model_bytes) / 1024:.1f} KB")

n_features = len(feature_names)
model = SimpleNetwork(
    input_dim=n_features,
    act=torch.nn.LeakyReLU(negative_slope=0.01),
    hidden_layer_dims=[HIDDEN_LAYER_DIM] * HIDDEN_LAYERS,
    output_dim=1,
)
model.load_state_dict(torch.load(io.BytesIO(model_bytes), map_location='cpu', weights_only=True))
model.eval()

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model loaded: {n_params:,} trainable parameters")
print(f"\nArchitecture:\n{model}")

## 5. Run inference on all samples

For each sample, the prediction pipeline is:

1. **Forward pass** — feed the 18 features through the MLP to get a raw correction value.
2. **Bound with tanh** — apply `tanh` so the correction stays in (−1, 1), preventing wild extrapolation.
3. **Add to baseline** — scale the exponential-atmosphere density to log-space and add the NN correction.
4. **Unscale** — convert back from normalized log-space to physical density (kg/m³).

We measure accuracy using **Mean Absolute Percentage Error (MAPE)**.

In [ ]:
predictions = []
ground_truths = []
expo_baselines = []
mapes = []

with torch.no_grad():
    for s in samples:
        features = torch.tensor(s['instantaneous_features']).unsqueeze(0)  # (1, 18)
        expo_atm = torch.tensor(s['exponential_atmosphere'])
        gt = s['ground_truth']

        # Step 1 & 2: NN forward pass + tanh bounding
        raw_correction = model(features).squeeze()
        bounded_correction = torch.tanh(raw_correction)

        # Step 3: Add correction to scaled baseline
        scaled_baseline = scale_density(expo_atm, log_min, log_max)
        scaled_prediction = scaled_baseline + bounded_correction

        # Step 4: Convert back to physical density
        density_pred = unscale_density(scaled_prediction, log_min, log_max).item()

        predictions.append(density_pred)
        ground_truths.append(gt)
        expo_baselines.append(s['exponential_atmosphere'])
        mapes.append(abs(density_pred - gt) / gt * 100)

# Print results table
print(f"{'Sample':>6}  {'Predicted [kg/m³]':>20}  {'Ground Truth [kg/m³]':>22}  {'MAPE [%]':>10}")
print("-" * 65)
for i, (pred, gt, mape) in enumerate(zip(predictions, ground_truths, mapes)):
    print(f"{i:>6}  {pred:>20.6e}  {gt:>22.6e}  {mape:>10.2f}")
print("-" * 65)
print(f"Mean Absolute Percentage Error: {sum(mapes) / len(mapes):.2f}%")

## 6. Visualize results

### Predicted vs. ground truth

A perfect model would place all points on the diagonal line. We also plot the exponential-atmosphere baseline to show how much the NN correction improves things.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

ax.scatter(ground_truths, expo_baselines, alpha=0.6, label='Exponential atmosphere (baseline)', marker='x', color='gray')
ax.scatter(ground_truths, predictions, alpha=0.7, label='Karman NN prediction', marker='o', color='tab:blue', edgecolors='k', linewidths=0.3)

# Perfect-prediction diagonal
lo = min(min(ground_truths), min(predictions)) * 0.8
hi = max(max(ground_truths), max(predictions)) * 1.2
ax.plot([lo, hi], [lo, hi], 'k--', lw=1, label='Perfect prediction')

ax.set_xlabel('Ground truth density [kg/m³]')
ax.set_ylabel('Predicted density [kg/m³]')
ax.set_title('Predicted vs. Ground Truth Density')
ax.legend()
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

### Per-sample MAPE

This bar chart shows the percentage error for each sample. Lower is better.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(range(len(mapes)), mapes, color='tab:orange', edgecolor='k', linewidth=0.3)
ax.axhline(sum(mapes) / len(mapes), color='tab:red', ls='--', lw=1.5, label=f'Mean MAPE = {sum(mapes)/len(mapes):.1f}%')
ax.set_xlabel('Sample index')
ax.set_ylabel('MAPE [%]')
ax.set_title('Per-Sample Absolute Percentage Error')
ax.legend()
plt.tight_layout()
plt.show()